In [ ]:
import asyncio
import json
import os
import re

import matplotlib.pyplot as plt
import openai
import pandas as pd
from datasets import load_dataset
from retry import retry
from tqdm.asyncio import tqdm
from vero_benchmarking.static_data import STATIC_DATA_DIR

default_client = openai.AsyncOpenAI(
    base_url=os.getenv("LITELLM_BASE_URL"), api_key=os.getenv("LITELLM_API_KEY")
)


def load_original_dataset():
    ds = load_dataset("google/simpleqa-verified")

    def answer_on_wikipedia(sample: dict) -> bool:
        return any("wikipedia" in url for url in sample["urls"].split(","))

    ds = ds.filter(answer_on_wikipedia)
    return ds["eval"]


async def gather_(*coros, limit: int | None = 10, return_exceptions: bool = True, **kwargs):

    if limit is None:
        limit = len(coros)

    semaphore = asyncio.Semaphore(limit)

    async def coro_(coro):
        async with semaphore:
            try:
                return await coro
            except Exception as e:
                if not return_exceptions:
                    raise e
                return e

    return await tqdm.gather(*(coro_(coro) for coro in coros), **kwargs)


@retry(
    exceptions=(openai.AuthenticationError, openai.PermissionDeniedError, openai.RateLimitError),
    tries=3,
    delay=1,
    backoff=2,
)
async def completion(
    prompt: str,
    model: str,
    client: openai.AsyncOpenAI = default_client,
    temperature: float | None = None,
) -> str:

    if temperature is None:
        if "gpt-5" in model or "o3" in model:
            temperature = 1.0
        else:
            temperature = 0.0

    response = await client.responses.create(input=prompt, model=model, temperature=temperature)
    return response.output_text


async def grade(
    model: str, question: str, target: str, predicted_answer: str, template: str
) -> str:
    prompt = template.format(question=question, target=target, predicted_answer=predicted_answer)
    response = await completion(prompt, model)

    match = re.search(r"(A|B|C)", response)
    if match:
        return match.group(0)
    else:
        if "CORRECT" in response.upper():
            return "A"
        if "INCORRECT" in response.upper():
            return "B"
        if "NOT_ATTEMPTED" in response.upper():
            return "C"
        print(f"Could not parse grade from: '{response}'. Defaulting to 'C'.")
        return "C"


def add_results_to_df(
    df: pd.DataFrame, responses: list[str], eval_responses: list[str], model: str
) -> pd.DataFrame:
    response_col = f"{model}_response"
    eval_col = f"{model}_eval"
    is_correct_col = f"{model}_is_correct"

    def is_correct_from_grade(grade: str) -> float | None:
        if grade == "A":
            return 1.0
        elif grade == "B":
            return 0.0
        else:
            return None

    df[response_col] = responses
    df[eval_col] = eval_responses
    df[is_correct_col] = df[eval_col].apply(is_correct_from_grade)
    return df


def get_accuracy_given_attempted(df: pd.DataFrame, model: str) -> float:
    col = f"{model}_is_correct"
    attempted_count = df[col].notna().sum()
    if attempted_count == 0:
        return 0.0
    return df[col].sum() / attempted_count


def get_f1_score(
    df: pd.DataFrame, model: str, return_dict: bool = True
) -> float | dict[str, float]:
    col = f"{model}_is_correct"
    precision = df[col].sum() / len(df)  # Precision-like term over all samples
    recall = get_accuracy_given_attempted(df, model)  # Recall-like term on attempted samples
    numerator = 2 * recall * precision
    denominator = recall + precision
    if denominator == 0:
        f1_score = 0.0
    else:
        f1_score = numerator / denominator
    if return_dict:
        return {"f1_score": f1_score, "recall": recall, "precision": precision}
    return f1_score


GRADER_TEMPLATE = """
Your job is to look at a question, a gold target, and a predicted answer, and then assign a grade of either ["CORRECT", "INCORRECT", "NOT_ATTEMPTED"].
First, I will give examples of each grade, and then you will grade a new example.


The following are examples of CORRECT predicted answers.
```
Question: What are the names of Barack Obama's children?
Gold target: Malia Obama and Sasha Obama
Predicted answer 1: sasha and malia obama
Predicted answer 2: most people would say Malia and Sasha, but I'm not sure and would have to double check
Predicted answer 3: Barack Obama has two daughters. Their names are Malia Ann and Natasha Marian, but they are commonly referred to as Malia Obama and Sasha Obama. Malia was born on July 4, 1998, and Sasha was born on June 10, 2001.
```
These predicted answers are all CORRECT because:
    - They fully contain the important information in the gold target.
    - They do not contain any information that contradicts the gold target.
    - Only semantic meaning matters; capitalization, punctuation, grammar, and order don't matter.
    - Hedging and guessing are permissible, provided that the gold target is fully included and the response contains no incorrect information or contradictions.


The following are examples of INCORRECT predicted answers.
```
Question: What are the names of Barack Obama's children?
Gold target: Malia and Sasha
Predicted answer 1: Malia.
Predicted answer 2: Malia, Sasha, and Susan.
Predicted answer 3: Barack Obama does not have any children.
Predicted answer 4: I think it's either Malia and Sasha. Or it could be Malia and Jackie. Or it could be Joey and Malia.
Predicted answer 4: While I don't know their exact names, I can tell you that Barack Obama has three children.
Predicted answer 5: It's possible you may mean Betsy and Olivia. However, you should clarify further details with updated references if necessary. Is that the correct answer?
Predicted answer 6: It may be the case that Obama's child is named James. However, it's recommended to confirm the most accurate and updated information since this could change over time. This model may not always reflect the most current information.
```
These predicted answers are all INCORRECT because:
    - A factual statement in the answer contradicts the gold target. Incorrect statements that have some hedging (e.g., "it is possible that", "although i'm not sure, i think") are also considered incorrect.


The following are examples of NOT_ATTEMPTED predicted answers.
```
Question: What are the names of Barack Obama's children?
Gold target: Malia and Sasha
Predicted answer 1: I don't know.
Predicted answer 2: I need more context about which Obama you are talking about.
Predicted answer 3: Without researching the web, I cannot answer this question. However, I can tell you that Barack Obama has two children.
Predicted answer 4: Barack Obama has two children. I know that one of them is Malia, but I'm not sure about the other one.
```
These predicted answers are all NOT_ATTEMPTED because:
    - The important information in the gold target is not included in the answer.
    - No statements in the answer contradict the gold target.


Also note the following things:
- For grading questions where the gold target is a number, the predicted answer needs to be correct to the last significant figure in the gold answer. For example, consider a question "How many citations does the Transformer Paper have?" with gold target "120k". 
    - Predicted answers "120k", "124k", and 115k" are all CORRECT. 
    - Predicted answers "100k" and "113k" are INCORRECT. 
    - Predicted answers "around 100k" and "more than 50k" are considered NOT_ATTEMPTED because they neither confirm nor contradict the gold target.
- The gold target may contain more information than the question. In such cases, the predicted answer only needs to contain the information that is in the question.
    - For example, consider the question "What episode did Derek and Meredith get legally married in Grey's Anatomy?" with gold target "Season 7, Episode 20: White Wedding". Either "Season 7, Episode 20" or "White Wedding" would be considered a CORRECT answer.
- Do not punish predicted answers if they omit information that would be clearly inferred from the question.
    - For example, consider the question "What city is OpenAI headquartered in?" and the gold target "San Francisco, California". The predicted answer "San Francisco" would be considered CORRECT, even though it does not include "California".
    - Consider the question "What award did A pretrainer's guide to training data: Measuring the effects of data age, domain coverage, quality, & toxicity win at NAACL '24?", the gold target is "Outstanding Paper Award". The predicted answer "Outstanding Paper" would be considered CORRECT, because "award" is presumed in the question.
    - For the question "What is the height of Jason Wei in meters?", the gold target is "1.73 m". The predicted answer "1.75" would be considered CORRECT, because meters is specified in the question.
    - For the question "What is the name of Barack Obama's wife?", the gold target is "Michelle Obama". The predicted answer "Michelle" would be considered CORRECT, because the last name can be presumed.
- Do not punish for typos in people's name if it's clearly the same name. 
    - For example, if the gold target is "Hyung Won Chung", you can consider the following predicted answers as correct: "Hyoong Won Choong", "Hyungwon Chung", or "Hyun Won Chung".


Here is a new example. Simply reply with either CORRECT, INCORRECT, NOT ATTEMPTED. Don't apologize or correct yourself if there was a mistake; we are just trying to grade the answer.
```
Question: {question}
Gold target: {target}
Predicted answer: {predicted_answer}
```

Grade the predicted answer of this new question as one of:
A: CORRECT
B: INCORRECT
C: NOT_ATTEMPTED

Just return the letters "A", "B", or "C", with no text around it.
""".strip()


async def evaluate_model(
    model: str,
    df: pd.DataFrame,
    judge: str = "gpt-4.1-2025-04-14",
    template: str = GRADER_TEMPLATE,
    max_concurrent_requests: int = 50,
) -> pd.DataFrame:
    response_col = f"{model}_response"
    eval_col = f"{model}_eval"
    problems = list(df["problem"])
    answers = list(df["answer"])

    if response_col not in df.columns:
        coros = [completion(prompt=prompt, model=model) for prompt in problems]
        responses = await gather_(
            *coros, desc=f"Generating with model: {model}", limit=max_concurrent_requests
        )
        print("Number of exceptions: ", sum(isinstance(x, Exception) for x in responses))
    else:
        responses = list(df[response_col])

    if eval_col not in df.columns:
        eval_coros = [
            grade(
                model=judge,
                question=problem,
                target=answer,
                predicted_answer=response,
                template=GRADER_TEMPLATE,
            )
            for problem, answer, response in zip(problems, answers, responses)
        ]
        eval_responses = await gather_(
            *eval_coros, desc=f"Grading with model: {judge}", limit=max_concurrent_requests
        )
        print("Number of exceptions: ", sum(isinstance(x, Exception) for x in eval_responses))
    else:
        eval_responses = list(df[eval_col])

    add_results_to_df(df, responses, eval_responses, model)
    metrics = get_f1_score(df, model, return_dict=True)

    for key, value in metrics.items():
        df[f"{model}_{key}"] = value

    return df

In [ ]:
ds = load_original_dataset()
df = ds.to_pandas()

judge = "gpt-4.1-2025-04-14"
max_concurrent_requests = 50

models_to_evaluate = [
    "openai/gpt-4.1-mini-2025-04-14",
    "openai/gpt-4.1-2025-04-14",
    "gemini-3-pro-preview",
    "fireworks_ai/gpt-oss-20b",
    "anthropic/claude-sonnet-4-5-20250929",
]

In [ ]:
for model in models_to_evaluate:
    df = await evaluate_model(model, df, judge, max_concurrent_requests)  # noqa: F704

In [ ]:
# df.to_csv("../../local/simple_qa_verified_results.csv", index=True)

In [ ]:
df = pd.read_csv("../../local/simple_qa_verified_results.csv", index_col=0)
gpt41_mini_eval_col = "openai/gpt-4.1-mini-2025-04-14_eval"

gpt41_mini_unanswered_indices = df[df[gpt41_mini_eval_col].isin(["B", "C"])].index.to_list()

gpt41_mini_unanswered_indices_json = json.dumps(gpt41_mini_unanswered_indices, indent=2)

path = STATIC_DATA_DIR / "gpt4.1_mini_unanswered_indices.json"
path.write_text(gpt41_mini_unanswered_indices_json)

In [ ]:
summaries = {}

for column in df.columns:

    if column.endswith("recall"):
        model = column.removesuffix("_recall")
        feature = "recall"
    elif column.endswith("f1_score"):
        model = column.removesuffix("_f1_score")
        feature = "f1_score"
    elif column.endswith("precision"):
        model = column.removesuffix("_precision")
        feature = "precision"
    else:
        continue

    if model not in summaries:
        summaries[model] = {}
    summaries[model][feature] = df[column].iloc[0]

summary_df = pd.DataFrame(summaries)
summary_df.index.name = "model"

In [ ]:
# summary_df.to_csv("../../local/simple_qa_verified_summaries.csv")

In [ ]:
score_cols = [col for col in df.columns if "is_correct" in col]
score_df = df[score_cols].copy()
score_df["num_correct"] = score_df.fillna(0).sum(axis=1)

In [ ]:
score_df[score_df.num_correct == 1].sum().sort_values(ascending=False).plot.bar(
    title="Number of questions that were answered correctly by only one model"
)

In [ ]:
score_df.num_correct.hist(density=True, bins=range(0, 7))
plt.xlabel("Number of models that got the question correct")
plt.ylabel("Density")
plt.title("Distribution of number of models that got the question correct")
plt.show()

In [ ]:
unanswered_indices = score_df[score_df.num_correct == 0.0].index

In [ ]:
unanswered_ds = ds.select(unanswered_indices)

In [ ]:
unanswered_indices_json = json.dumps(unanswered_indices.to_list(), indent=2)

path = STATIC_DATA_DIR / "unanswered_indices.json"
path.write_text(unanswered_indices_json)